# UW-CycleGAN on Google Colab

รัน UW-CycleGAN (ฟื้นฟูภาพใต้น้ำ) บน Colab

**ก่อนเริ่ม:** เมนู `Runtime → Change runtime type → T4 GPU` (โค้ดนี้ต้องใช้ GPU)

ลำดับ: 1) ติดตั้ง → 2) เตรียม weights → 3) Test → 4) Train (ถ้าต้องการ)

## 1. Setup

In [ ]:
!nvidia-smi -L

In [ ]:
%cd /content
!git clone -b UW-Colab https://github.com/nantapongpa-a11y/UW-CycleGAN.git
%cd /content/UW-CycleGAN

In [ ]:
# backbone EfficientNet-Lite3 ที่โมเดลใช้ (แทน torch.hub path เดิมของผู้เขียน)
!pip install -q git+https://github.com/rwightman/gen-efficientnet-pytorch.git

In [ ]:
# เก็บ weights และผลลัพธ์ไว้ใน Google Drive จะได้ไม่หายเมื่อ session หลุด
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/UW-CycleGAN'
import os
os.makedirs(f'{DRIVE_DIR}/model', exist_ok=True)

## 2. เตรียม weights ของโมเดล

ไฟล์ weights ที่ผู้เขียนเทรนไว้อยู่บน Baidu (ดู `model/notes`):
https://pan.baidu.com/s/1zAGyhGS2tXpGNsZbwP1aZg  รหัส `9f38`

ดาวน์โหลดมาแล้วอัปโหลดเข้า Google Drive ที่ `MyDrive/UW-CycleGAN/model/` ให้มี 4 ไฟล์:

```
model_t.pt  model_b.pt  model_A.pt  model_d.pt
```

In [ ]:
import glob
!mkdir -p model
!cp -v {DRIVE_DIR}/model/*.pt model/ 2>/dev/null

missing = [f for f in ['model_t.pt', 'model_b.pt', 'model_A.pt', 'model_d.pt'] if not os.path.exists(f'model/{f}')]
print('ขาดไฟล์:', missing if missing else 'ไม่มี พร้อมใช้งาน')

## 3. Test

ค่าเริ่มต้นใช้รูปตัวอย่างใน `test_img/data1/img`
ถ้าอยากใช้รูปของตัวเอง ให้รันเซลล์ถัดไปแล้วเลือกไฟล์ (ข้ามได้)

In [ ]:
# (ไม่บังคับ) อัปโหลดรูปใต้น้ำของตัวเอง
from google.colab import files
TEST_DIR = 'test_img/my/img'
os.makedirs(TEST_DIR, exist_ok=True)
for name, data in files.upload().items():
    with open(os.path.join(TEST_DIR, name), 'wb') as f:
        f.write(data)
print(os.listdir(TEST_DIR))

In [ ]:
# ถ้าไม่ได้อัปโหลดรูป จะใช้รูปตัวอย่าง
TEST_DIR = 'test_img/my/img' if glob.glob('test_img/my/img/*') else 'test_img/data1/img'
RESULT_DIR = f'{DRIVE_DIR}/test_result'
print('test on:', TEST_DIR)

!MPLBACKEND=Agg python test_cycle.py     --weights-t ./model/model_t.pt --weights-b ./model/model_b.pt     --weights-A ./model/model_A.pt --weights-d ./model/model_d.pt     --underwater-dir {TEST_DIR} --clean-dir {TEST_DIR} --depth-dir {TEST_DIR}     --img-size 256 --GPU 0     --project {RESULT_DIR} --name test01

In [ ]:
# แสดงผล: ภาพใต้น้ำ (input) เทียบกับภาพที่ฟื้นฟูแล้ว
import matplotlib.pyplot as plt
from PIL import Image

outs = sorted(glob.glob(f'{RESULT_DIR}/test01/weights/*_ours.png'))
for out in outs:
    under = out.replace('_ours.png', '_under.png')
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(Image.open(under)); ax[0].set_title('underwater')
    ax[1].imshow(Image.open(out)); ax[1].set_title('UW-CycleGAN')
    for a in ax: a.axis('off')
    plt.show()
print(f'ผลลัพธ์ทั้งหมดอยู่ที่ {RESULT_DIR}/test01/weights')

## 4. Train (ไม่บังคับ)

ใช้ dataset ตัวอย่างใน `./dataset` (air_image + air_depth `.mat`, underwater_img + underwater_refer)
ถ้ามี dataset เต็ม (NYU + UIEB) ให้วางใน Drive แล้วเปลี่ยน path ด้านล่าง

- เทรนจากศูนย์: ไม่ต้องใส่ `--weights_*`
- fine-tune จาก weights ของผู้เขียน: ใส่ `--weights_t ./model/model_t.pt ...` (ตัวอย่างคอมเมนต์ไว้ด้านล่าง)
- weights จะถูกเซฟทุก 100 epoch ที่ `runs/exp1/weights/` ใน Drive

Colab ฟรีตัด session ประมาณ 12 ชม. ถ้าหลุด ให้ต่อด้วย `--strat_epoch <epoch>` และชี้ `--weights_*` ไปที่ไฟล์ล่าสุด

In [ ]:
RUNS_DIR = f'{DRIVE_DIR}/runs'

!MPLBACKEND=Agg python train_cycle.py     --clean-dir ./dataset/air_image --depth-dir ./dataset/air_depth     --underwater-dir ./dataset/underwater_img --underwater-dir-refer ./dataset/underwater_refer     --epochs 201 --batch-size 2 --img-size 256 --GPU 0     --project {RUNS_DIR} --name exp1
#   --weights_t ./model/model_t.pt --weights_b ./model/model_b.pt #   --weights_A ./model/model_A.pt --weights_d ./model/model_d.pt